# Phase 1 Notebook: Mention Detection
Reads text archives from data/01_input and writes only to data/10_mention_detection.

## 1) Project Setup
Resolve repository paths and make the source package importable in this notebook session.

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / 'data').exists() and (cur / 'speakermining' / 'src').exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise RuntimeError('Repository root not found.')

ROOT = find_repo_root(Path.cwd())
SRC = ROOT / 'speakermining' / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ROOT

## 2) Input Discovery
Build the list of archive text files that will be parsed into structured episode blocks.

In [ ]:
from process.mention_detection.config import DEFAULT_PDF_TXT_INPUTS, ZDF_ARCHIVE_DIR, PHASE_DIR
from process.text_extraction.text import load_episode_blocks_from_many

input_paths = [ROOT / ZDF_ARCHIVE_DIR / name for name in DEFAULT_PDF_TXT_INPUTS]
input_paths

## 3) Episode And Season Extraction
Parse text blocks into episode records, then derive season-level aggregates from extracted episode metadata.

In [ ]:
from process.mention_detection.episode import extract_episode_and_publication_rows, save_episodes
from process.mention_detection.publications import save_publications
from process.mention_detection.season import extract_season_rows, save_seasons
from process.mention_detection.duplicates import filter_exact_duplicates_with_report

episode_blocks = load_episode_blocks_from_many(input_paths)
episodes_raw_df, publications_raw_df = extract_episode_and_publication_rows(episode_blocks)
seasons_raw_df = extract_season_rows(episodes_raw_df)

episodes_df, episodes_dup_df, episodes_dup_path, episodes_dup_stats = filter_exact_duplicates_with_report(
    "episodes", episodes_raw_df, ROOT / PHASE_DIR
)
publications_df, publications_dup_df, publications_dup_path, publications_dup_stats = filter_exact_duplicates_with_report(
    "publications", publications_raw_df, ROOT / PHASE_DIR
)
seasons_df, seasons_dup_df, seasons_dup_path, seasons_dup_stats = filter_exact_duplicates_with_report(
    "seasons", seasons_raw_df, ROOT / PHASE_DIR
)

ep_path = save_episodes(episodes_df, ROOT / PHASE_DIR)
pu_path = save_publications(publications_df, ROOT / PHASE_DIR)
se_path = save_seasons(seasons_df, ROOT / PHASE_DIR)

print(f"episodes: raw={episodes_dup_stats['raw_rows']} kept={episodes_dup_stats['kept_rows']} exact_duplicates={episodes_dup_stats['duplicate_rows']} -> {ep_path}")
print(f"publications: raw={publications_dup_stats['raw_rows']} kept={publications_dup_stats['kept_rows']} exact_duplicates={publications_dup_stats['duplicate_rows']} -> {pu_path}")
print(f"seasons: raw={seasons_dup_stats['raw_rows']} kept={seasons_dup_stats['kept_rows']} exact_duplicates={seasons_dup_stats['duplicate_rows']} -> {se_path}")
print(f"duplicate files: {episodes_dup_path.name}, {publications_dup_path.name}, {seasons_dup_path.name}")

## 4) Mention Extraction
Extract person and topic mentions from the same episode text blocks.
Institution extraction is deferred to the semantic disambiguation phase (with Wikidata context).


In [ ]:
from process.mention_detection.guest import extract_person_mentions, save_person_mentions
from process.mention_detection.topic import extract_topic_mentions, save_topic_mentions
from process.mention_detection.duplicates import filter_exact_duplicates_with_report

# Extract persons and topics
persons_raw_df = extract_person_mentions(episode_blocks, episodes_df)
topics_raw_df = extract_topic_mentions(episode_blocks, episodes_df)

persons_df, persons_dup_df, persons_dup_path, persons_dup_stats = filter_exact_duplicates_with_report(
    "persons", persons_raw_df, ROOT / PHASE_DIR
)
topics_df, topics_dup_df, topics_dup_path, topics_dup_stats = filter_exact_duplicates_with_report(
    "topics", topics_raw_df, ROOT / PHASE_DIR
)

pe_path = save_person_mentions(persons_df, ROOT / PHASE_DIR)
to_path = save_topic_mentions(topics_df, ROOT / PHASE_DIR)

print(f"persons: raw={persons_dup_stats['raw_rows']} kept={persons_dup_stats['kept_rows']} exact_duplicates={persons_dup_stats['duplicate_rows']} -> {pe_path}")
print(f"topics: raw={topics_dup_stats['raw_rows']} kept={topics_dup_stats['kept_rows']} exact_duplicates={topics_dup_stats['duplicate_rows']} -> {to_path}")
print(f"duplicate files: {persons_dup_path.name}, {topics_dup_path.name}")


## ToDo
* Infer gender from Description (Schauspielerin = w, 95 % certainty, "female occupation ending"; Schauspieler = m, 60 % certainty; Ehemann = m, 100 % certainty; Ehefrau = w, 100 % certainty, etc.)
  * There will be ambigous labels, such as "Comedian" or "Vorstand"
* Ensure guests and mentions are stored differently - e.g. Franjo Pooth was mentioned in an episode within the guest block, but only because his wife was there, not he himself. Similarly, names will occur in the topic section. So there are at least three different kinds of name mentionings:
  * Guests, the most important ones, usually highlighted by Firstname LASTNAME (Last name all capitalized)
  * Topic mentions, humans mentioned as part of the topics, e.g. Obama
  * Other mentions, not further specified

## 5) Quantitative Validation
Check key metrics and consistency signals across all extracted tables before moving to candidate generation.

### Overview

In [ ]:
import pandas as pd

persons_conf = pd.to_numeric(persons_df["confidence"], errors="coerce")
topics_conf = pd.to_numeric(topics_df["confidence"], errors="coerce")

overview = pd.DataFrame(
    [
        {
            "table": "seasons",
            "rows": len(seasons_df),
            "unique_seasons": seasons_df["season_id"].nunique(),
        },
        {
            "table": "episodes",
            "rows": len(episodes_df),
            "unique_episodes": episodes_df["episode_id"].nunique(),
            "missing_publication_links": int(episodes_df["publikation_id"].isna().sum()),
            "missing_dates": int(episodes_df["publikationsdatum"].isna().sum()),
        },
        {
            "table": "publications",
            "rows": len(publications_df),
            "unique_publications": publications_df["publikation_id"].nunique(),
            "linked_episodes": publications_df["episode_id"].nunique(),
            "primary_publications": int((publications_df["is_primary"].astype(int) == 1).sum()),
        },
        {
            "table": "persons",
            "rows": len(persons_df),
            "unique_episodes": persons_df["episode_id"].nunique(),
            "unique_names": persons_df["name"].nunique(),
            "avg_confidence": round(float(persons_conf.mean()), 3),
        },
        {
            "table": "topics",
            "rows": len(topics_df),
            "unique_episodes": topics_df["episode_id"].nunique(),
            "unique_topics": topics_df["topic"].nunique(),
            "avg_confidence": round(float(topics_conf.mean()), 3),
        },
    ]
).fillna("")

season_dates = seasons_df.assign(
    start=pd.to_datetime(seasons_df["start_time"], errors="coerce", dayfirst=True),
    end=pd.to_datetime(seasons_df["end_time"], errors="coerce", dayfirst=True),
)

season_summary = {
    "season_count": len(seasons_df),
    "min_episodes_per_season": int(seasons_df["episode_count"].min()),
    "median_episodes_per_season": float(seasons_df["episode_count"].median()),
    "max_episodes_per_season": int(seasons_df["episode_count"].max()),
    "season_start": season_dates["start"].min().date().isoformat(),
    "season_end": season_dates["end"].max().date().isoformat(),
}

display(overview)
season_summary

### Episodes

In [ ]:
episode_dates = pd.to_datetime(episodes_df["publikationsdatum"], errors="coerce", dayfirst=True)

episodes_per_season = (
    episodes_df.groupby("season", dropna=False)["episode_id"]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
    .rename("episodes")
)

episode_validation = {
    "episodes_total": len(episodes_df),
    "unique_episode_ids": episodes_df["episode_id"].nunique(),
    "duplicate_episode_ids": int(episodes_df["episode_id"].duplicated().sum()),
    "date_coverage_start": episode_dates.min().date().isoformat(),
    "date_coverage_end": episode_dates.max().date().isoformat(),
    "missing_season": int(episodes_df["season"].isna().sum()),
    "missing_staffel": int(episodes_df["staffel"].isna().sum()),
    "missing_folge": int(episodes_df["folge"].isna().sum()),
    "missing_folgennr": int(episodes_df["folgennr"].isna().sum()),
}

publications_per_episode = publications_df.groupby("episode_id")["publikation_id"].nunique()
publication_link_stats = {
    "episodes_with_publication_rows": int(publications_per_episode.index.nunique()),
    "episodes_without_publication_rows": int(len(episodes_df) - publications_per_episode.index.nunique()),
    "publications_per_episode_min": int(publications_per_episode.min()),
    "publications_per_episode_median": float(publications_per_episode.median()),
    "publications_per_episode_max": int(publications_per_episode.max()),
}

display(episodes_df)
episode_validation, publication_link_stats, episodes_per_season

### Publications

In [ ]:
pub_dates = pd.to_datetime(publications_df["date"], errors="coerce", dayfirst=True)

publications_validation = {
    "rows_total": len(publications_df),
    "unique_publication_ids": publications_df["publikation_id"].nunique(),
    "duplicate_publication_ids": int(publications_df["publikation_id"].duplicated().sum()),
    "linked_episode_count": publications_df["episode_id"].nunique(),
    "primary_rows": int((publications_df["is_primary"].astype(int) == 1).sum()),
    "non_primary_rows": int((publications_df["is_primary"].astype(int) == 0).sum()),
    "date_start": pub_dates.min().date().isoformat(),
    "date_end": pub_dates.max().date().isoformat(),
    "missing_time": int(publications_df["time"].isna().sum()),
    "missing_duration": int(publications_df["duration"].isna().sum()),
}

program_distribution = (
    publications_df["program"]
    .fillna("<missing>")
    .value_counts()
    .rename_axis("program")
    .to_frame("rows")
)

display(publications_df)
publications_validation, program_distribution

### Persons

In [ ]:
persons_conf = pd.to_numeric(persons_df["confidence"], errors="coerce")
persons_per_episode = persons_df.groupby("episode_id")["mention_id"].nunique()

person_validation = {
    "mention_rows": len(persons_df),
    "unique_mention_ids": persons_df["mention_id"].nunique(),
    "duplicate_mention_ids": int(persons_df["mention_id"].duplicated().sum()),
    "episodes_with_person_mentions": int(persons_df["episode_id"].nunique()),
    "unique_person_names": int(persons_df["name"].nunique()),
    "avg_mentions_per_episode": round(float(persons_per_episode.mean()), 2),
    "median_mentions_per_episode": float(persons_per_episode.median()),
    "max_mentions_in_one_episode": int(persons_per_episode.max()),
    "avg_confidence": round(float(persons_conf.mean()), 3),
    "p10_confidence": round(float(persons_conf.quantile(0.10)), 3),
    "p50_confidence": round(float(persons_conf.quantile(0.50)), 3),
    "p90_confidence": round(float(persons_conf.quantile(0.90)), 3),
    "missing_description": int(persons_df["beschreibung"].isna().sum()),
    "invalid_confidence_rows": int(persons_conf.isna().sum()),
}

top_persons = (
    persons_df["name"]
    .value_counts()
    .head(10)
    .rename_axis("name")
    .to_frame("mentions")
)

person_rule_distribution = (
    persons_df["parsing_rule"]
    .value_counts()
    .rename_axis("parsing_rule")
    .to_frame("rows")
)

display(persons_df)
person_validation, top_persons, person_rule_distribution

### Topics

In [ ]:
topics_conf = pd.to_numeric(topics_df["confidence"], errors="coerce")
topics_per_episode = topics_df.groupby("episode_id")["mention_id"].nunique()

topic_validation = {
    "mention_rows": len(topics_df),
    "unique_mention_ids": topics_df["mention_id"].nunique(),
    "duplicate_mention_ids": int(topics_df["mention_id"].duplicated().sum()),
    "episodes_with_topic_mentions": int(topics_df["episode_id"].nunique()),
    "unique_topic_labels": int(topics_df["topic"].nunique()),
    "avg_topics_per_episode": round(float(topics_per_episode.mean()), 2),
    "median_topics_per_episode": float(topics_per_episode.median()),
    "max_topics_in_one_episode": int(topics_per_episode.max()),
    "avg_confidence": round(float(topics_conf.mean()), 3),
    "p10_confidence": round(float(topics_conf.quantile(0.10)), 3),
    "p50_confidence": round(float(topics_conf.quantile(0.50)), 3),
    "p90_confidence": round(float(topics_conf.quantile(0.90)), 3),
    "invalid_confidence_rows": int(topics_conf.isna().sum()),
}

top_topics = (
    topics_df["topic"]
    .value_counts()
    .head(10)
    .rename_axis("topic")
    .to_frame("mentions")
)

topic_rule_distribution = (
    topics_df["parsing_rule"]
    .value_counts()
    .rename_axis("parsing_rule")
    .to_frame("rows")
)

display(topics_df)
topic_validation, top_topics, topic_rule_distribution